# Deliverable 3 · 25 Preguntas Difíciles en 5 Bloques (150 respuestas)

Conjunto complementario de 25 preguntas difíciles (IDs 101–125) preparadas por el equipo
para contrastar razonamiento normativo avanzado, colisión léxica y cálculos ponderados.

- 25 preguntas × 6 condiciones = **150 respuestas**.
- Ejecutado en **5 bloques de 30 respuestas** (5 preguntas × 6 condiciones por bloque).
- Cada bloque o cada 6 respuestas genera un snapshot inmutable para no perder avance.
- No altera el lote de 50 preguntas ya completado y auditado.

## 1. Instalar dependencias fijadas

In [ ]:
import sys, subprocess, importlib.metadata
PINNED = {'transformers':'4.57.6', 'accelerate':'1.11.0', 'bitsandbytes':'0.48.1',
          'sentence-transformers':'5.1.2', 'tokenizers':'0.22.2'}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                      *[p+'=='+v for p,v in PINNED.items()]])
for module, package in [(p.replace('-', '_'), p) for p in PINNED] + [
    ('numpy','numpy'), ('scipy','scipy'), ('torch','torch'), ('sklearn','scikit-learn'),
    ('huggingface_hub','huggingface-hub')]:
    loaded = sys.modules.get(module)
    if loaded is not None and hasattr(loaded, '__version__'):
        if loaded.__version__ != importlib.metadata.version(package):
            raise RuntimeError('Reinicia la sesión: versión anterior en memoria de ' + package)
print('Dependencias instaladas. No se cargó ningún modelo todavía.')

## 2. Restaurar código, prompts y las 25 preguntas difíciles

In [ ]:
BUNDLE_B64 = (
    '...'
)
EXPECTED = '726daff2fceb6cbfa37d6c28bab5bebb52fa42f1d4c23bacb35d5372a4334047'
FILE_HASHES = {'d3_agentes/__init__.py': 'ee0695533a98ad593db63822f7d3775fe565e1c4a5575f78de54034987d44d9d', 'd3_agentes/bridge.py': '0e15d6153defdb4bc77c0c2a99d32e28e34a31b5f24c97b94fd0e06c23924ca1', 'd3_agentes/common.py': '76f9cb1d436e46f095d37a70ddd9b1ab8c8da0f8e1ce5ee1f8f2f1831077bb20', 'd3_agentes/core.py': '98d335c11552512b03dac92eb0d4c0b280719fca5cc82f75fe1f3cd0f3370018', 'd3_agentes/downloads.py': '9345bffa531e1fb55dcd6d572f8afbc9c324972e29c88103aa297486042924cc', 'd3_agentes/experiment.py': 'd3c6b4b1ed314258e175df95193411782ad63ff39d077fb5b5e723a65c4cc8c8', 'd3_agentes/runtime.py': '112401003b7051b2f462f0496bfa1f6a313a7fa01c7c7a262930837fbc04ab6e', 'd3_agentes/tools.py': 'f075926974535b9b03aa8eaf431b0a803af13d81444b08f136818643a09ddab0', 'prompts/auditor.txt': 'b71aed834269f72d691c13b81b375bf1c64b0356749547f491daf6efb81058db', 'prompts/completitud.txt': '72918aa2f8a5bfb7981073823e44145c2574691812672487df6664b540e9c538', 'prompts/descomposicion.txt': 'cf09766bec652248002dfa204b2cf8fff4df6330f22f01c838b1e1c2d4de50f8', 'prompts/investigador.txt': '0bcc8694aa10ecdcfa5c17438071a7390cd77a2a355141191e13b5704ee1c9ea', 'prompts/redactor.txt': '25df34e7d299ddaa25db0f2877dba7965b5129b48ca64e6d92cf7dddb546ef33', 'prompts/verificacion.txt': '8aa50bb8a3fbe4949331186e65ae4a09711be095a04af3830035cec74b1a1587', 'd2_runtime/experimento.py': 'd20ce66a54c7791a480e0d9466e1740b34a8c5198df94ec52c1d4868219e22d0', 'd2_runtime/sistema.py': 'be9ff8714f6580b30618ef4d45c98e744463da620e98721701dadfe9521fe130', 'd2_runtime/recuperador_portable.py': 'a04c720b0cf25eb8010996dd28f70f8a08806b6c556e0be6bc2ea3a6fb4dd056', 'data/corpus.json': '0a31ff812a1969ff81bbc895feaba490e3bd027b092e52101296e3c872719400', 'data/fragmentos_busqueda.json': 'c125615c1c3cddb23d69c3b2b93d7a73e076c90fe861cc217b59f293aa9b14cd', 'data/preguntas.json': '70f89171c04b52a70980ac3fc318b0e27973ea8421c7425ab7da7160f4a2f154', 'data/configuracion.json': 'd5dd7936c24d811b302cc961b926735a8626708a94fc65b21424dad03eaab310'}

import base64, gzip, hashlib, json, sys
from pathlib import Path
packed = base64.b64decode(BUNDLE_B64, validate=True)
if hashlib.sha256(packed).hexdigest() != EXPECTED:
    raise ValueError('Paquete alterado.')
sources = json.loads(gzip.decompress(packed))
if set(sources) != set(FILE_HASHES):
    raise ValueError('Inventario del paquete incorrecto.')
WORKDIR = Path.cwd() / ('d3_dificiles_runtime_' + EXPECTED[:16])
for name, text in sources.items():
    path = Path(name)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError('Ruta no permitida.')
    if hashlib.sha256(text.encode()).hexdigest() != FILE_HASHES[name]:
        raise ValueError('Fuente alterada: ' + name)
    target = WORKDIR / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding='utf-8')
previous = sys.modules.get('d3_agentes')
if previous is not None and Path(previous.__file__).resolve().parent != (WORKDIR / 'd3_agentes').resolve():
    raise RuntimeError('Reinicia la sesión para cargar esta versión de D3.')
sys.path.insert(0, str(WORKDIR.resolve())) if str(WORKDIR.resolve()) not in sys.path else None
print('Fuentes restauradas en:', WORKDIR.name)

## 3. Cargar modelos e inicializar runtime

In [ ]:
import sys
from pathlib import Path
import torch
from d3_agentes.runtime import load

USAR_DRIVE = False
if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT = Path('/content/drive/MyDrive/GenIA_D3_Agentes/runs_dificiles')
    except Exception:
        OUTPUT = Path('/content/d3_dificiles_runs')
else:
    OUTPUT = Path('/content/d3_dificiles_runs')

OUTPUT.mkdir(parents=True, exist_ok=True)
ENV = load(OUTPUT)
print('Runtime listo. Preguntas difíciles cargadas:', len(ENV['questions']))

## 4. Ejecutar por bloques de 30 respuestas (5 bloques de 30 = 150 respuestas)

In [ ]:
# Ejecutar en Colab después de las secciones 1, 2 y 3.
# Lote de 25 preguntas difíciles (150 respuestas = 5 bloques de 30 respuestas).
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, urllib.request, shutil
from d3_agentes.experiment import audit_zip, prepare, export
from d3_agentes.common import Limits, PromptBook, sealed_load, VARIANTS
import d3_agentes.experiment as experiment_module
from d3_agentes.runtime import execute

class BloqueTerminado(BaseException):
    """Parada operativa normal después de exportar una respuesta completa."""

def preparar_corrida_dificiles(ENV, OUTPUT):
    """Prepara o recupera el directorio de ejecución para las 25 preguntas difíciles."""
    config_esperada = {
        'preguntas': ENV['questions'],
        'variantes': list(VARIANTS),
        'limites': Limits(),
        'etiqueta': 'dificiles-25'
    }
    RUN_DIR, actual = prepare(ENV['questions'], ENV['retriever'], ENV['settings'], OUTPUT,
                              VARIANTS, Limits(), 'dificiles-25', PromptBook())
    export(RUN_DIR)
    print(f'Directorio de corrida: {RUN_DIR.name}', flush=True)
    return RUN_DIR

def ejecutar_bloque_dificil(ENV, OUTPUT, RUN_DIR, descargar, tamaño=30, total=150):
    """Ejecuta un bloque de 30 respuestas (5 preguntas x 6 variantes) y exporta snapshot."""
    original_export = experiment_module.export
    original_export(RUN_DIR)
    zip_path = RUN_DIR.with_suffix('.zip')
    inicial = audit_zip(zip_path)['completadas'] if zip_path.exists() else 0

    if inicial >= total:
        print(f'Las {total} respuestas ya están conservadas. No se ejecuta inferencia.')
        return RUN_DIR

    objetivo = min((inicial // tamaño + 1) * tamaño, total)
    numero = objetivo // tamaño
    print(f'======================================================================')
    print(f'BLOQUE {numero}/5: {inicial}/{total} conservadas; objetivo {objetivo}/{total}.', flush=True)
    print(f'Se generarán {objetivo - inicial} respuestas pendientes sin repetir las guardadas.', flush=True)
    print(f'======================================================================', flush=True)

    backup_progress = {'n': inicial}

    def export_bloque(folder):
        archive_path = original_export(folder)
        resumen_path = Path(folder) / 'resumen_tecnico.json'
        if not resumen_path.exists():
            return archive_path
        n = json.loads(resumen_path.read_text())['completadas']
        if n >= backup_progress['n'] + 6 or n >= objetivo:
            snapshot = Path(folder).parent / (Path(folder).name + f'_respaldo_{n:03}de{total}.zip')
            shutil.copy2(archive_path, snapshot)
            try:
                descargar(str(snapshot))
                print(f'--> DESCARGA SOLICITADA: {n}/{total} ({snapshot.name})', flush=True)
            except Exception as exc:
                print(f'Descargar desde Archivos de Colab: {snapshot.name} ({type(exc).__name__})', flush=True)
            backup_progress['n'] = n
        if n >= objetivo:
            raise BloqueTerminado()
        return archive_path

    experiment_module.export = export_bloque
    try:
        execute(ENV, ENV['questions'], OUTPUT, variants=VARIANTS, tag='dificiles-25', verbose=False)
    except BloqueTerminado:
        print(f'BLOQUE {numero}/5 TERMINADO ({objetivo}/{total}).', flush=True)
    finally:
        experiment_module.export = original_export
        original_export(RUN_DIR)
        progress = audit_zip(RUN_DIR.with_suffix('.zip'))
        print(f'RESPALDO VERIFICADO: {progress["completadas"]}/{progress["esperadas"]}', flush=True)
        if progress['completadas'] < objetivo:
            try:
                descargar(str(RUN_DIR.with_suffix('.zip')))
            except Exception as exc:
                print('Descargar respaldo parcial desde Archivos:', type(exc).__name__, flush=True)
    return RUN_DIR

CONTINUAR_HASTA_FINAL = True  # True: ejecuta automáticamente los 5 bloques hasta 150

if __name__ == '__main__':
    from google.colab import files
    RUN_DIR = preparar_corrida_dificiles(ENV, OUTPUT)
    if globals().get('CONTINUAR_HASTA_FINAL', False):
        while True:
            RUN_DIR = ejecutar_bloque_dificil(ENV, OUTPUT, RUN_DIR, files.download, tamaño=30, total=150)
            progress = audit_zip(RUN_DIR.with_suffix('.zip'))
            if progress['completadas'] >= 150:
                print('¡ÉXITO TOTAL! 150/150 respuestas de preguntas difíciles completadas y respaldadas.', flush=True)
                break
    else:
        RUN_DIR = ejecutar_bloque_dificil(ENV, OUTPUT, RUN_DIR, files.download, tamaño=30, total=150)

## 5. Descargar o recuperar respaldos

In [ ]:
from d3_agentes.downloads import prepare_download
ZIP = prepare_download(OUTPUT, None)
if ZIP is not None:
    print('ZIP listo para descargar:', ZIP.resolve(), '| bytes:', ZIP.stat().st_size)
    try:
        from google.colab import files
        files.download(str(ZIP))
    except Exception as exc:
        print('Descarga automática no iniciada. Descarga manualmente desde el panel Archivos.')